In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# 0) Imports
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from pyspark.sql import functions as F

# 1) Configurações
BASE = "https://erddap.emodnet.eu"

# Seu URL (relativo) -> vamos compor o absoluto
RELATIVE_URL = (
    "/erddap/griddap/V2023_chemistry_bca5_368a_2f92.csv?Water_body_dissolved_inorganic_nitrogen_deepest%5B(2014-02-16T00:00:00Z):1:(2019-11-16T00:00:00Z)%5D%5B(36.25):1:(42.0)%5D%5B(-2.25):1:(4.625)%5D"
)

URL = f"{BASE}{RELATIVE_URL}"

TARGET_TABLE = "emodnet_din_deepest"   # nome da tabela no Lakehouse

print("Fetching:", URL)

# 2) Leitura do CSV do ERDDAP
# ERDDAP geralmente inclui linhas de comentário/metadados começando com '#'
df = pd.read_csv(URL, comment="#")

print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

# 3) Normalização de colunas
# Em muitos griddap CSV, as colunas vêm como: time, latitude, longitude, <variavel>
# Mas às vezes aparecem com nomes ligeiramente diferentes. Vamos padronizar.

# Remover espaços e aspas “estranhas”
df.columns = [c.strip().replace('"', "") for c in df.columns]

# Tentar identificar colunas essenciais
# - time: pode ser 'time' ou algo parecido
# - latitude: 'latitude'
# - longitude: 'longitude'
# - variável: Water_body_dissolved_inorganic_nitrogen_deepest
time_candidates = [c for c in df.columns if c.lower() in ("time", "date", "datetime")]
lat_candidates  = [c for c in df.columns if c.lower() == "latitude"]
lon_candidates  = [c for c in df.columns if c.lower() == "longitude"]

var_name = "Water_body_dissolved_inorganic_nitrogen_deepest"
var_candidates = [c for c in df.columns if c == var_name]

if not time_candidates:
    raise ValueError(f"Não encontrei coluna de tempo. Colunas: {df.columns.tolist()}")
if not lat_candidates or not lon_candidates:
    raise ValueError(f"Não encontrei latitude/longitude. Colunas: {df.columns.tolist()}")
if not var_candidates:
    raise ValueError(f"Não encontrei a variável {var_name}. Colunas: {df.columns.tolist()}")

time_col = time_candidates[0]
lat_col  = lat_candidates[0]
lon_col  = lon_candidates[0]
var_col  = var_candidates[0]

# Padronizar nomes
df = df.rename(columns={
    time_col: "time",
    lat_col: "latitude",
    lon_col: "longitude",
    var_col: "DIN_deepest"
})

# 4) Tipagem e limpeza
# Parse de tempo
df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True)

# Converter coordenadas e variável para numérico
for c in ["latitude", "longitude", "DIN_deepest"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Remover linhas inválidas
df = df.dropna(subset=["time", "latitude", "longitude", "DIN_deepest"]).copy()

# Adicionar colunas úteis
df["year"] = df["time"].dt.year.astype(int)
df["ingested_at_utc"] = datetime.now(timezone.utc).isoformat()

print("Clean shape:", df.shape)
df.head()

# 5) Converter para Spark e gravar no Lakehouse (Delta table)
sdf = spark.createDataFrame(df)

# (Opcional) deduplicar (por tempo+lat+lon)
sdf = sdf.dropDuplicates(["time", "latitude", "longitude"])

# Gravar como tabela Delta no Lakehouse
# mode="append" para incremental; use "overwrite" se quiser recriar
(
    sdf.write
      .mode("append")
      .format("delta")
      .saveAsTable(TARGET_TABLE)
)

print(f"✅ Gravado no Lakehouse: {TARGET_TABLE}")

# 6) Checagem rápida
spark.table(TARGET_TABLE).orderBy(F.col("time").desc()).show(10, truncate=False)


StatementMeta(, 47e7c4e7-c798-48e3-bdb4-d72f3991269c, 3, Finished, Available, Finished, False)

Clean shape: (38976, 6)
✅ Gravado no Lakehouse: emodnet_din_deepest
+-------------------+--------+---------+-----------+----+--------------------------------+
|time               |latitude|longitude|DIN_deepest|year|ingested_at_utc                 |
+-------------------+--------+---------+-----------+----+--------------------------------+
|2019-11-16 00:00:00|36.25   |-2.25    |7.373688   |2019|2026-03-31T12:19:38.068545+00:00|
|2019-11-16 00:00:00|36.25   |-2.125   |7.3786426  |2019|2026-03-31T12:19:38.068545+00:00|
|2019-11-16 00:00:00|36.25   |-2.0     |7.3831005  |2019|2026-03-31T12:19:38.068545+00:00|
|2019-11-16 00:00:00|36.25   |-1.875   |7.3870444  |2019|2026-03-31T12:19:38.068545+00:00|
|2019-11-16 00:00:00|36.25   |-1.75    |7.390599   |2019|2026-03-31T12:19:38.068545+00:00|
|2019-11-16 00:00:00|36.25   |-1.625   |7.393964   |2019|2026-03-31T12:19:38.068545+00:00|
|2019-11-16 00:00:00|36.25   |-1.5     |7.397142   |2019|2026-03-31T12:19:38.068545+00:00|
|2019-11-16 00:00:00|3